# Plot saved randomized-encoding QPINN results

Run this notebook after `03_train_randomized_encoding_save_data.ipynb`.  It reads local data from `data/qpinn_b2_randomized_encoding_data.npz` and writes only the original randomized-encoding figures to `figures/`.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile

import numpy as np

_MPL_CACHE = Path(tempfile.gettempdir()) / "matplotlib-cache"
_MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPL_CACHE))
_XDG_CACHE = Path(tempfile.gettempdir()) / "xdg-cache"
_XDG_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(_XDG_CACHE))

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt


def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    markers = (
        "01_train_and_save_data.ipynb",
        "02_plot_saved_data.ipynb",
        "03_train_randomized_encoding_save_data.ipynb",
        "04_plot_randomized_encoding_saved_data.ipynb",
        "requirements.txt",
    )
    for candidate in [cwd, *cwd.parents]:
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    return cwd


def show_or_close_current_figure() -> None:
    if "ipykernel" in sys.modules:
        plt.show()
    else:
        plt.close()


PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "data"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "qpinn_b2_randomized_encoding_data.npz"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Run 03_train_randomized_encoding_save_data.ipynb first: {DATA_PATH}")

saved_data = np.load(DATA_PATH)
model_names = [str(value) for value in saved_data["model_names"]]
RANDOMIZED_MODEL_SLUGS = {
    "ordinary QFM": "ordinary_qfm",
    "random sign-flip QFM": "random_signflip_qfm",
    "random B2 QFM": "random_b2_qfm",
}

histories = {}
predictions = {}
errors = {}
symmetry = {}
for model_name in model_names:
    slug = RANDOMIZED_MODEL_SLUGS[model_name]
    histories[model_name] = {
        "loss": saved_data[f"{slug}_loss"],
        "residual": saved_data[f"{slug}_residual"],
        "boundary": saved_data[f"{slug}_boundary"],
        "theta_grad_norm": saved_data[f"{slug}_theta_grad_norm"],
    }
    predictions[model_name] = saved_data[f"{slug}_prediction_grid"]
    errors[model_name] = float(saved_data[f"{slug}_error"])
    symmetry[model_name] = {
        "sign_flip_mse": float(saved_data[f"{slug}_sign_flip_mse"]),
        "permutation_mse": float(saved_data[f"{slug}_permutation_mse"]),
    }

grid_x1 = saved_data["grid_x1"]
grid_x2 = saved_data["grid_x2"]
target_grid = saved_data["target_grid"]

print(f"loaded plot-ready randomized-encoding data: {DATA_PATH}")
print(f"writing figures to: {FIGURE_DIR}")


In [ ]:
plt.figure(figsize=(7.0, 4.2))
for model_name, history in histories.items():
    plt.semilogy(history["loss"], label=model_name)
plt.xlabel("training step")
plt.ylabel("QPINN loss")
plt.title("Randomized-encoding PSR QPINN training")
plt.legend(frameon=False)
plt.tight_layout()
training_path = FIGURE_DIR / "qpinn_b2_training_loss.png"
plt.savefig(training_path, dpi=220)
show_or_close_current_figure()
print(f"saved {training_path}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.0, 7.0), constrained_layout=True)
panels = [("analytical solution", target_grid)] + list(predictions.items())

vmin = min(np.min(panel) for _, panel in panels)
vmax = max(np.max(panel) for _, panel in panels)

for ax, (title, values) in zip(axes.ravel(), panels):
    image = ax.contourf(grid_x1, grid_x2, values, levels=40, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")

fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.82, label="$u(x_1,x_2)$")
solution_path = FIGURE_DIR / "qpinn_b2_solution_comparison.png"
plt.savefig(solution_path, dpi=220)
show_or_close_current_figure()
print(f"saved {solution_path}")


In [ ]:
labels = list(symmetry.keys())
sign_values = [symmetry[name]["sign_flip_mse"] for name in labels]
perm_values = [symmetry[name]["permutation_mse"] for name in labels]

xpos = np.arange(len(labels))
width = 0.36

plt.figure(figsize=(8.0, 4.2))
plt.bar(xpos - width / 2, sign_values, width, label="sign flip")
plt.bar(xpos + width / 2, perm_values, width, label="coordinate permutation")
plt.yscale("log")
plt.xticks(xpos, labels, rotation=15, ha="right")
plt.ylabel("mean squared defect")
plt.title("Symmetry of the randomized-encoding estimator")
plt.legend(frameon=False)
plt.tight_layout()
symmetry_path = FIGURE_DIR / "qpinn_b2_symmetry_metrics.png"
plt.savefig(symmetry_path, dpi=220)
show_or_close_current_figure()
print(f"saved {symmetry_path}")
